[Reference](https://medium.com/codetodeploy/how-to-build-an-ai-agent-harness-2-0-and-engineer-better-than-99-of-developers-028e4fc01b50?sk=e074eeb6c9326fff585809c4986ecef4$0)

```
                 ┌─────────────────┐
                 │    AI Agent     │
                 └────────┬────────┘
                          │
       ┌──────────────────┼──────────────────┐
       ↓                  ↓                  ↓
    Context              Tools          Constraints
       ↓                  ↓                  ↓
    Memory            Execution        Verification
       ↓                  ↓                  ↓
    Rules              Feedback          Guardrails
       └──────────────────┼──────────────────┘
                          ↓
                   Agent Harness
```

```
Task
 ↓
Context
 ↓
Agent
 ↓
Tools
 ↓
Code
 ↓
Verification
 ↓
Feedback
 ↺
```

```
my-project/
├── AGENTS.md
├── src/
│   ├── controllers/
│   ├── services/
│   ├── repositories/
│   └── models/
├── tests/
└── package.json

```

In [1]:
tools = [
    read_file,
    search_code,
    list_files,
    apply_patch,
    run_tests,
    run_linter,
    run_typecheck,
]

def run_tests():
    result = subprocess.run(
        ["npm", "test"],
        capture_output=True,
        text=True
    )
  return {
        "success": result.returncode == 0,
        "stdout": result.stdout,
        "stderr": result.stderr
    }

def search_code(query):
    result = subprocess.run(
        ["rg", query, "src"],
        capture_output=True,
        text=True
    )
   return result.stdout


def get_context(task):
    relevant_files = search_code(
        extract_keywords(task)
    )

  architecture = read_file(
        "docs/architecture.md"
    )
    rules = read_file(
        "AGENTS.md"
    )
    return {
        "task": task,
        "rules": rules,
        "architecture": architecture,
        "relevant_files": relevant_files
    }

```
agent-memory/
├── decisions.md
├── progress.md
├── failures.md
└── architecture.md
```

```
# decisions.md

## Customer API
We use repository classes for all database access.
Controllers should never call Prisma directly.

Reason:
Keeps persistence concerns separate from
business logic and makes services easier to test.
```

```
# failures.md

## Customer API
Previous implementation attempted to access
Prisma directly from the controller.
Rejected because this violates the repository pattern.
```

In [2]:
def verify():
    checks = [
        run_tests(),
        run_typecheck(),
        run_linter(),
    ]
    return all(
        check["success"]
        for check in checks
    )


def run_agent(task):
    context = get_context(task)
    for attempt in range(3):
        action = agent.decide(context)
        result = execute(action)
        context.append(result)
        if task_complete(result):
            verification = verify()
            if verification:
                return "Task completed"
            context.append(
                "Verification failed. Fix the issues."
            )
    return "Task failed after 3 attempts"

```
 Developer
                             │
                             ↓
                          Task
                             │
                             ↓
                    ┌─────────────────┐
                    │  Agent Harness  │
                    └────────┬────────┘
                             │
        ┌────────────────────┼────────────────────┐
        ↓                    ↓                    ↓
     Context               Memory              Rules
        ↓                    ↓                    ↓
     Retrieval           Decisions           Guardrails
        └────────────────────┼────────────────────┘
                             ↓
                           Agent
                             │
                             ↓
                           Tools
                             │
                             ↓
                       Code Changes
                             │
                             ↓
                       Verification
                             │
                    ┌────────┴────────┐
                    ↓                 ↓
                  FAIL              PASS
                    │                 │
                    ↓                 ↓
                 Feedback           Done
                    │
                    └──────→ Agent
```

In [4]:
class AgentHarness:
    def __init__(self, agent):
        self.agent = agent
        self.memory = []
        self.max_attempts = 3
    def run(self, task):
        context = self.load_context(task)
        for attempt in range(self.max_attempts):
            action = self.agent.decide(
                task=task,
                context=context
            )
            result = self.execute(action)
            context.append(result)
            verification = self.verify()
            if verification.success:
                self.save_memory(context)
                return "Success"
            context.append({
                "type": "verification_error",
                "message": verification.error
            })
        return "Failed safely"